### Ячейка 1 — импорт модулей

In [ ]:
from __future__ import annotations

import logging
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Dict
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from edlm_search.agent.candidate import Candidate
from edlm_search.agent.llm_pipeline import LLMPipeline
from edlm_search.agent.llm_search_agent import AgentCandidateRecord
from edlm_search.agent.llm_search_agent import LLMSearchRun
from edlm_search.agent.llm_search_agent import build_problem
from edlm_search.agent.llm_search_agent import create_llm_pipeline
from edlm_search.agent.llm_search_agent import run_llm_search_for_all_datasets
from edlm_search.agent.problem import Problem
from edlm_search.agent.runner import UnsafeRunner
from edlm_search.experiments.datasets import load_ett_csv_dataset

### Ячейка 2 — глобальные переменные и конфигурация окружения

In [ ]:
# Определяем корневую директорию проекта.
current_dir = Path.cwd().resolve()
repo_root_candidates = (current_dir, current_dir.parent)
repo_root: Path | None = None
for candidate_path in repo_root_candidates:
    if (candidate_path / 'src' / 'edlm_search').is_dir():
        repo_root = candidate_path
        break
if repo_root is None:
    raise RuntimeError('Не удалось найти корень репозитория с каталогом "src/edlm_search".')

src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Загружаем .env при наличии.
env_path = repo_root / '.env'
if env_path.is_file():
    load_dotenv(dotenv_path=env_path)

# Настраиваем логирование.
logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
)
LOGGER = logging.getLogger('etth_llm_agent_experiments')

# Пути к датасетам.
ett_data_dir = src_dir / 'ETDataset' / 'ETT-small'
dataset_paths: Dict[str, Path] = {
    'ETTh1': ett_data_dir / 'ETTh1.csv',
    'ETTh2': ett_data_dir / 'ETTh2.csv',
}

# Общая конфигурация эксперимента.
dataset_names: List[str] = list(dataset_paths.keys())
# max_rows_options: List[int] = [5000, 10000, 20000]
max_rows_options: List[int] = [5000]

train_ratio = float(os.getenv('TRAIN_RATIO', '0.8'))
target_column = os.getenv('TARGET_COLUMN', 'OT')
plots_max_points = 500
statement_path = src_dir / 'statement.md'

# Параметры LLM-агента.
llm_metric_name = os.getenv('AGENT_METRIC_NAME', os.getenv('LLM_METRIC_NAME', 'mse'))
llm_num_epochs = int(os.getenv('AGENT_NUM_EPOCHS', os.getenv('LLM_NUM_EPOCHS', '5')))
llm_num_initial_candidates = int(
        os.getenv('AGENT_NUM_INITIAL_CANDIDATES', os.getenv('LLM_NUM_INITIAL_CANDIDATES', '2')))
llm_num_crossover_candidates = int(
        os.getenv('AGENT_NUM_CROSSOVER_CANDIDATES', os.getenv('LLM_NUM_CROSSOVER_CANDIDATES', '1')))

# Параметры провайдера LLM.
llm_provider_name_raw = os.getenv('CHAT_CLIENT_PROVIDER', os.getenv('LLM_PROVIDER', 'lmstudio'))
llm_provider_name = llm_provider_name_raw.strip().lower()

lm_studio_base_url = os.getenv('LM_STUDIO_BASE_URL', 'http://127.0.0.1:1234/v1')
lm_studio_model_name = os.getenv('LM_STUDIO_MODEL_NAME', 'your-lmstudio-model-name')
lm_studio_temperature = float(os.getenv('LM_STUDIO_TEMPERATURE', '0.1'))
lm_studio_top_p = float(os.getenv('LM_STUDIO_TOP_P', '0.95'))

deepseek_base_url = os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com/v1')
deepseek_model_name = os.getenv('DEEPSEEK_MODEL_NAME', 'deepseek-reasoner')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY', '')
deepseek_temperature = float(os.getenv('DEEPSEEK_TEMPERATURE', '0.1'))
deepseek_top_p = float(os.getenv('DEEPSEEK_TOP_P', '0.95'))

openai_base_url = os.getenv('OPENAI_BASE_URL', 'https://api.openai.com/v1')
openai_model_name = os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')
openai_api_key = os.getenv('OPENAI_API_KEY', '')
openai_temperature = float(os.getenv('OPENAI_TEMPERATURE', '0.6'))
openai_top_p = float(os.getenv('OPENAI_TOP_P', '0.85'))

# Директория для сохранения артефактов ноутбука.
artifacts_output_dir = (repo_root / 'artifacts' / 'llm_notebook_runs').resolve()
artifacts_output_dir.mkdir(parents=True, exist_ok=True)

LOGGER.info(f'Корень репозитория: {repo_root}')
LOGGER.info(f'Каталог с данными ETT: {ett_data_dir}')
LOGGER.info(f'Каталог артефактов ноутбука: {artifacts_output_dir}')
LOGGER.info(f'LLM-провайдер: {llm_provider_name}')

## Конфигурации и структуры данных

Далее определяем вспомогательные структуры, которые будут отвечать за:
- описание конфигурации провайдера и поиска;
- хранение параметров нарезки датасета;
- организацию результатов и предсказаний;
- вспомогательные классы для запуска LLM-агента и построения таблиц.

### Ячейка 4 — структуры и вспомогательные классы

In [ ]:
@dataclass
class NotebookLLMProviderConfig:
    """Конфигурация LLM-провайдера для ноутбука."""
    provider: str
    base_url: str
    model_name: str
    temperature: float
    top_p: float
    api_key: str | None


@dataclass
class NotebookLLMSearchConfig:
    """Конфигурация поиска LLM-агента."""
    num_initial_candidates: int
    num_crossover_candidates: int
    num_epochs_per_candidate: int
    metric_name: str


@dataclass(frozen=True)
class DatasetSplitConfig:
    """Описание конкретного запуска: датасет + ограничение по MAX_ROWS."""
    name: str
    csv_path: Path
    max_rows: int
    train_ratio: float


@dataclass(frozen=True)
class LLMRunKey:
    """Уникальный идентификатор прогона LLM-агента."""
    dataset_name: str
    max_rows: int


@dataclass
class PredictionPayload:
    """Набор данных для визуализации."""
    run_key: LLMRunKey
    dataset_name: str
    train_df: pd.DataFrame
    valid_df: pd.DataFrame
    train_predictions: np.ndarray
    valid_predictions: np.ndarray
    best_candidate_id: int
    metrics: Dict[str, float]


class DatasetLoader:
    """Отвечает за загрузку и разбиение датасетов ETT."""

    def split(self, config: DatasetSplitConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
        LOGGER.info(
                f'Загрузка датасета "{config.name}" с ограничением {config.max_rows} строк.'
        )
        if not config.csv_path.is_file():
            raise FileNotFoundError(f'Файл датасета не найден по пути {config.csv_path}')
        train_df, valid_df = load_ett_csv_dataset(
                csv_path=str(config.csv_path),
                max_rows=config.max_rows,
                train_ratio=config.train_ratio,
        )
        train_df = train_df.reset_index(drop=True)
        valid_df = valid_df.reset_index(drop=True)
        LOGGER.info(
                f'Датасет "{config.name}" загружен: train={len(train_df)}, valid={len(valid_df)}.'
        )
        return train_df, valid_df


class CandidateSelector:
    """Инкапсулирует выбор лучшего кандидата по метрике."""

    @staticmethod
    def select_best(records: List[AgentCandidateRecord], metric_name: str) -> AgentCandidateRecord:
        successful_records = [
            record for record in records
            if record.metrics is not None and metric_name in record.metrics
        ]
        if not successful_records:
            raise RuntimeError('Не найдено успешных кандидатов для выбора лучшего решения.')
        best_record = min(
                successful_records,
                key=lambda record: record.metrics[metric_name],
        )
        LOGGER.info(
                f'Лучший кандидат: id={best_record.candidate_id}, '
                f'{metric_name}={best_record.metrics[metric_name]:.6f}.'
        )
        return best_record


class CandidatePredictionCollector:
    """Выполняет запуск кандидата и извлекает предсказания последней эпохи."""

    def __init__(self, num_epochs: int):
        self._num_epochs = num_epochs

    async def collect(self, candidate: Candidate, train_df: pd.DataFrame, eval_df: pd.DataFrame) -> np.ndarray:
        runner = UnsafeRunner()
        run_args = {'num_epochs': self._num_epochs}
        latest_predictions: np.ndarray | None = None
        async for event in runner.run(
                candidate=candidate,
                train_df=train_df,
                validation_df=eval_df,
                run_args=run_args,
        ):
            if 'epoch_result' in event:
                latest_predictions = np.asarray(event['epoch_result'], dtype=np.float64).reshape(-1)
        if latest_predictions is None:
            raise RuntimeError('Кандидат завершил работу без предсказаний.')
        return latest_predictions


class LLMAgentExperimentExecutor:
    """Оборачивает запуск поиска LLM-агента для конкретного датасета."""

    def __init__(
            self,
            problem: Problem,
            llm_pipeline: LLMPipeline,
            search_config: NotebookLLMSearchConfig,
            provider_config: NotebookLLMProviderConfig,
            target_column_name: str,
    ) -> None:
        self._problem = problem
        self._llm_pipeline = llm_pipeline
        self._search_config = search_config
        self._provider_config = provider_config
        self._target_column_name = target_column_name

    async def run_for_split(
            self,
            dataset_config: DatasetSplitConfig,
            train_df: pd.DataFrame,
            valid_df: pd.DataFrame,
    ) -> tuple[LLMSearchRun, AgentCandidateRecord]:
        dataset_descriptor = SimpleNamespace(name=dataset_config.name)
        run = await run_llm_search_for_all_datasets(
                dataset_configs=[dataset_descriptor],
                train_dfs={dataset_config.name: train_df},
                valid_dfs={dataset_config.name: valid_df},
                problem=self._problem,
                llm_pipeline=self._llm_pipeline,
                config=self._search_config,
                target_column=self._target_column_name,
                provider_config=self._provider_config,
        )
        records = run.results.get(dataset_config.name, [])
        best_record = CandidateSelector.select_best(records, self._search_config.metric_name)
        return run, best_record


class LLMAgentResultsRegistry:
    """Хранит результаты всех запусков и готовит агрегированные представления."""

    def __init__(self) -> None:
        self._runs: Dict[LLMRunKey, LLMSearchRun] = {}
        self._train_frames: Dict[LLMRunKey, pd.DataFrame] = {}
        self._valid_frames: Dict[LLMRunKey, pd.DataFrame] = {}
        self._best_records: Dict[LLMRunKey, AgentCandidateRecord] = {}
        self._prediction_payloads: Dict[LLMRunKey, PredictionPayload] = {}
        self._split_configs: Dict[LLMRunKey, DatasetSplitConfig] = {}

    def register_run(
            self,
            run_key: LLMRunKey,
            split_config: DatasetSplitConfig,
            search_run: LLMSearchRun,
            train_df: pd.DataFrame,
            valid_df: pd.DataFrame,
            best_record: AgentCandidateRecord,
    ) -> None:
        self._runs[run_key] = search_run
        self._train_frames[run_key] = train_df.copy()
        self._valid_frames[run_key] = valid_df.copy()
        self._best_records[run_key] = best_record
        self._split_configs[run_key] = split_config
        LOGGER.info(
                f'Результаты прогона сохранены: dataset={run_key.dataset_name}, '
                f'max_rows={run_key.max_rows}, best_candidate={best_record.candidate_id}.'
        )

    def store_predictions(
            self,
            run_key: LLMRunKey,
            train_predictions: np.ndarray,
            valid_predictions: np.ndarray,
    ) -> None:
        best_record = self._best_records[run_key]
        payload = PredictionPayload(
                run_key=run_key,
                dataset_name=run_key.dataset_name,
                train_df=self._train_frames[run_key],
                valid_df=self._valid_frames[run_key],
                train_predictions=train_predictions,
                valid_predictions=valid_predictions,
                best_candidate_id=best_record.candidate_id,
                metrics=dict(best_record.metrics) if best_record.metrics is not None else {},
        )
        self._prediction_payloads[run_key] = payload
        LOGGER.info(
                f'Предсказания сохранены для dataset={run_key.dataset_name}, '
                f'max_rows={run_key.max_rows}.'
        )

    def list_run_keys(self) -> List[LLMRunKey]:
        keys = sorted(self._runs.keys(), key=lambda item: (item.dataset_name, item.max_rows))
        return keys

    def has_prediction_payload(self, run_key: LLMRunKey) -> bool:
        return run_key in self._prediction_payloads

    def get_train_df(self, run_key: LLMRunKey) -> pd.DataFrame:
        return self._train_frames[run_key].copy()

    def get_valid_df(self, run_key: LLMRunKey) -> pd.DataFrame:
        return self._valid_frames[run_key].copy()

    def get_best_record(self, run_key: LLMRunKey) -> AgentCandidateRecord:
        return self._best_records[run_key]

    def build_metrics_dataframe(self, metric_name: str) -> pd.DataFrame:
        rows: List[Dict[str, object]] = []
        for run_key in sorted(self._runs.keys(), key=lambda item: (item.dataset_name, item.max_rows)):
            record = self._best_records[run_key]
            metrics = record.metrics or {}
            row = {
                'dataset': run_key.dataset_name,
                'max_rows': run_key.max_rows,
                'best_candidate_id': record.candidate_id,
                metric_name: float(metrics.get(metric_name, float('nan'))),
                'wall_time_seconds': metrics.get('wall_time_seconds'),
                'total_energy_joules': metrics.get('total_energy_joules'),
                'fix_attempts': record.metadata.get('fix_attempts'),
                'total_input_tokens': record.metadata.get('total_input_tokens'),
                'total_output_tokens': record.metadata.get('total_output_tokens'),
            }
            rows.append(row)
        if not rows:
            return pd.DataFrame()
        return pd.DataFrame(rows)

    def iter_prediction_payloads(self) -> List[PredictionPayload]:
        ordered_keys = sorted(self._prediction_payloads.keys(), key=lambda item: (item.dataset_name, item.max_rows))
        return [self._prediction_payloads[key] for key in ordered_keys]


def build_llm_provider_config(
        provider_name: str,
        base_url: str,
        model_name: str,
        temperature_value: float,
        top_p_value: float,
        api_key_value: str | None,
) -> NotebookLLMProviderConfig:
    if not provider_name:
        raise ValueError('Имя провайдера LLM не может быть пустым.')
    if not base_url:
        raise ValueError('base_url провайдера должен быть задан.')
    if not model_name:
        raise ValueError('model_name провайдера должен быть задан.')
    provider_config = NotebookLLMProviderConfig(
            provider=provider_name,
            base_url=base_url.strip(),
            model_name=model_name.strip(),
            temperature=temperature_value,
            top_p=top_p_value,
            api_key=(api_key_value.strip() if api_key_value and api_key_value.strip() else None),
    )
    return provider_config

## Инициализация объектов и подготовка конфигураций

На этом шаге:
1. Строим конфигурации разбиений датасетов для всех комбинаций `(dataset, MAX_ROWS)`.
2. Создаём объекты `Problem`, `LLMPipeline`, исполнителя и хранилища результатов.
3. Настраиваем провайдера LLM на основе переменных окружения.

### Ячейка 6 — инициализация

In [ ]:
# Формируем список конфигураций разбиения.
dataset_split_configs: List[DatasetSplitConfig] = []
for dataset_name in dataset_names:
    csv_path = dataset_paths[dataset_name]
    for max_rows in max_rows_options:
        dataset_split_configs.append(
                DatasetSplitConfig(
                        name=dataset_name,
                        csv_path=csv_path,
                        max_rows=max_rows,
                        train_ratio=train_ratio,
                )
        )

# Определяем параметры провайдера согласно выбранному источнику.
if llm_provider_name == 'lmstudio':
    provider_config = build_llm_provider_config(
            provider_name='lmstudio',
            base_url=lm_studio_base_url,
            model_name=lm_studio_model_name,
            temperature_value=lm_studio_temperature,
            top_p_value=lm_studio_top_p,
            api_key_value=None,
    )
elif llm_provider_name == 'deepseek':
    provider_config = build_llm_provider_config(
            provider_name='deepseek',
            base_url=deepseek_base_url,
            model_name=deepseek_model_name,
            temperature_value=deepseek_temperature,
            top_p_value=deepseek_top_p,
            api_key_value=deepseek_api_key,
    )
elif llm_provider_name == 'openai':
    provider_config = build_llm_provider_config(
            provider_name='openai',
            base_url=openai_base_url,
            model_name=openai_model_name,
            temperature_value=openai_temperature,
            top_p_value=openai_top_p,
            api_key_value=openai_api_key,
    )
else:
    raise ValueError(f'Неизвестный провайдер LLM: {llm_provider_name}')

# Конфигурация поиска.
search_config = NotebookLLMSearchConfig(
        num_initial_candidates=llm_num_initial_candidates,
        num_crossover_candidates=llm_num_crossover_candidates,
        num_epochs_per_candidate=llm_num_epochs,
        metric_name=llm_metric_name,
)

# Подготовка остальных объектов.
problem = build_problem(statement_path)
llm_pipeline = create_llm_pipeline(provider_config)
dataset_loader = DatasetLoader()
results_registry = LLMAgentResultsRegistry()
prediction_collector = CandidatePredictionCollector(num_epochs=llm_num_epochs)
executor = LLMAgentExperimentExecutor(
        problem=problem,
        llm_pipeline=llm_pipeline,
        search_config=search_config,
        provider_config=provider_config,
        target_column_name=target_column,
)
LOGGER.info('Инициализация завершена, можно запускать эксперименты.')

## Запуск экспериментов LLM-агента

Следующая ячейка выполняет полный цикл:
1. Загружает каждый датасет с заданным `MAX_ROWS`.
2. Запускает LLM-агента на соответствующем сплите.
3. Сохраняет лучшего кандидата и его метрики.
4. Снова запускает кандидата для получения предсказаний на train/valid (для визуализации).

### Ячейка 8 — асинхронный запуск

In [ ]:
async def execute_all_runs() -> None:
    for split_config in dataset_split_configs:
        run_key = LLMRunKey(dataset_name=split_config.name, max_rows=split_config.max_rows)
        LOGGER.info(
                f'Запуск эксперимента: dataset={run_key.dataset_name}, max_rows={run_key.max_rows}.'
        )
        try:
            train_df, valid_df = dataset_loader.split(split_config)
        except Exception as exc:
            LOGGER.exception(
                    f'Не удалось подготовить данные для dataset={run_key.dataset_name}, '
                    f'max_rows={run_key.max_rows}: {exc}'
            )
            continue

        try:
            search_run, best_record = await executor.run_for_split(split_config, train_df, valid_df)
        except Exception as exc:
            LOGGER.exception(
                    f'Ошибка запуска LLM-агента для dataset={run_key.dataset_name}, '
                    f'max_rows={run_key.max_rows}: {exc}'
            )
            continue

        results_registry.register_run(run_key, split_config, search_run, train_df, valid_df, best_record)


await execute_all_runs()
LOGGER.info('Все доступные эксперименты завершены.')

## Подготовка предсказаний для визуализации

In [ ]:
async def ensure_prediction_payloads(
        registry: LLMAgentResultsRegistry,
        collector: CandidatePredictionCollector,
) -> None:
    for run_key in registry.list_run_keys():
        if registry.has_prediction_payload(run_key):
            continue
        LOGGER.info(
                f'Пересчёт предсказаний: dataset={run_key.dataset_name}, max_rows={run_key.max_rows}.'
        )
        best_record = registry.get_best_record(run_key)
        train_df = registry.get_train_df(run_key)
        valid_df = registry.get_valid_df(run_key)
        train_predictions = await collector.collect(best_record.candidate, train_df, train_df)
        valid_predictions = await collector.collect(best_record.candidate, train_df, valid_df)
        registry.store_predictions(run_key, train_predictions, valid_predictions)


await ensure_prediction_payloads(results_registry, prediction_collector)
LOGGER.info('Все необходимые наборы предсказаний готовы.')

## Сводка метрик по лучшим кандидатам

Построим таблицу, где для каждой комбинации `(dataset, MAX_ROWS)` приведены:
- идентификатор лучшего кандидата;
- значение целевой метрики `MSE`;
- измеренные wall-time и энергопотребление (если доступны);
- количество фиксаций и токенов.

### Ячейка 10 — таблица метрик

In [ ]:
metrics_df = results_registry.build_metrics_dataframe(llm_metric_name)
if metrics_df.empty:
    LOGGER.info('Нет данных для построения таблицы метрик.')
else:
    display(metrics_df)
    metrics_csv_path = artifacts_output_dir / 'llm_agent_metrics_summary.csv'
    metrics_df.to_csv(metrics_csv_path, index=False)
    LOGGER.info(f'Таблица метрик сохранена в {metrics_csv_path}')

## Функция построения графиков

Определим вспомогательную функцию для построения линейных графиков фактических и предсказанных значений.

### Ячейка 12 — функция визуализации

In [ ]:
def plot_series(values_true: np.ndarray, values_pred: np.ndarray, title: str, max_points: int) -> None:
    if values_true.ndim != 1 or values_pred.ndim != 1:
        raise ValueError('Ожидаются одномерные последовательности для построения графика.')
    series_length = min(len(values_true), len(values_pred))
    if series_length == 0:
        raise ValueError('Последовательности для графика не должны быть пустыми.')
    limit = min(series_length, max_points)
    index = np.arange(limit)
    plt.figure(figsize=(10, 4))
    plt.plot(index, values_true[:limit], label='y_true')
    plt.plot(index, values_pred[:limit], label='y_pred')
    plt.title(title)
    plt.xlabel('time index')
    plt.ylabel(target_column)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Визуализация временных рядов (train и valid)

Построим для каждого успешного запуска два графика:
1. Предсказания на обучающей части (train).
2. Предсказания на валидации (valid).

Это позволит наглядно оценить поведение лучших кандидатов.

### Ячейка 14 — построение графиков

In [ ]:
results_registry

In [ ]:
prediction_payloads = results_registry.iter_prediction_payloads()
if not prediction_payloads:
    LOGGER.info('Нет сохранённых предсказаний для визуализации.')
else:
    for payload in prediction_payloads:
        train_values = payload.train_df[target_column].to_numpy(dtype=np.float64)
        valid_values = payload.valid_df[target_column].to_numpy(dtype=np.float64)

        train_title = (
            f'{payload.dataset_name} | max_rows={payload.run_key.max_rows} | '
            f'train | candidate={payload.best_candidate_id}'
        )
        valid_title = (
            f'{payload.dataset_name} | max_rows={payload.run_key.max_rows} | '
            f'valid | candidate={payload.best_candidate_id}'
        )

        plot_series(train_values, payload.train_predictions, train_title, plots_max_points)
        plot_series(valid_values, payload.valid_predictions, valid_title, plots_max_points)

## Итоги

- Все выбранные комбинации `(dataset, MAX_ROWS)` обработаны с помощью LLM-агента.
- Сформирована таблица метрик лучших кандидатов и сохранена в `artifacts/llm_notebook_runs/`.
- Построены графики фактических и предсказанных значений для train/valid частей, что позволяет визуально оценить качество полученных решений.